In [1]:
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import io
import pathlib
import time
import copy
import numpy as np
import math
import tensorflow as tf
import keras
import rasterio
from rasterio.transform import Affine
from rasterio.merge import merge as rasterio_merge
from rasterio.warp import calculate_default_transform, reproject, Resampling

2026-11-24 10:37:30.912928: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-11-24 10:37:30.912986: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-11-24 10:37:31.047936: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [2]:
from google.cloud import firestore  # type:ignore[attr-defined]
from google.cloud import storage  # type:ignore[attr-defined]

from usl_models.flood_ml.dataset import compute_dem_sink_channel, compute_flow_features
# from usl_models.flood_ml.model import (
#     FloodModel,
#     SpatialAttention,
#     FloodConvLSTM,
#     GreenAmptGate,
# )
from usl_models.flood_ml import customloss, metastore
from usl_models.shared import downloader

/scratch/hw4402/conda_envs/test_new/lib/python3.11/site-packages/google/api_core/_python_version_support.py:273: FutureWarning: You are using a Python version (3.11.15) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2027-10-24). Please upgrade to the latest Python version, or at least Python 3.12, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [3]:
STUDY_AREA = "Chicago_Predictions"
RAINFALL_CONFIGS = [
    f"Chicago%2FRainfall_Data_{i}.txt" for i in range(1, 10)
] 

In [4]:
db = firestore.Client(project='climateiq-test')
gcs_client = storage.Client(project='climateiq-test')

In [ ]:
study_area_ref = db.collection("study_areas").document(STUDY_AREA)
study_area_doc = study_area_ref.get()